In [1]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import backtrader as bt
from openpyxl import load_workbook
import tempfile, os
from statsmodels.regression.rolling import RollingOLS
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import random
import pytz
import time
import os
from xbbg import blp
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, Dropdown, HBox, VBox, Button, Output, Text, widgets
import sympy as sp
from sklearn.metrics import r2_score
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from IPython import get_ipython
import matplotlib.dates as mdates
from pydataquery import DataQuery
import re
import statsmodels.api as sm
from scipy.optimize import minimize
import scipy.stats as stats
import itertools
import warnings
from statsmodels.tsa.seasonal import seasonal_decompose
import yfinance as yf
import csv
import uuid
from concurrent.futures import ThreadPoolExecutor
import warnings
from multiprocess import Pool
import time
warnings.filterwarnings('ignore')

In [13]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_excel("Z1.xlsx")[['Security','Description','Asset Class',
                               'Geography','Instrument Type','Asset Class Tag']]
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['Description'].apply(lambda x: x.split(' ')[-1]=='Z')].sort_values(by='Description').set_index('Security')
df = df.sort_values(by=['Asset Class', 'Geography','Instrument Type','Asset Class Tag'])

# x1 = blp.bdh(tickers=list(df.index), flds='px_last', start_date='2000-1-1')
x = x1.copy()
x.columns = [item[0] for item in x.columns]
x.index = pd.to_datetime(x.index)

# latest z-score per security (last non-NaN observation)
zsc = x.iloc[-1].round(1).rename('ZSc')

# half-life of mean reversion (in days) per security via OU/AR(1) fit
def half_life(series, dt=1):
    s = series.dropna()
    if len(s) < 3:
        return np.nan
    y = s.values[1:]
    x_lag = sm.add_constant(s.values[:-1])
    res = sm.OLS(y, x_lag).fit()
    alpha, beta = res.params
    if beta <= 0 or beta >= 1:        # no/insufficient mean reversion
        return np.nan
    kappa = -np.log(beta) / dt
    return np.log(2) / kappa

hl = x.apply(half_life).round(1).rename('HalfLife')

df = pd.concat([df, zsc, hl], axis=1).sort_values(by='HalfLife')  # shortest at top
df

,Description,Asset Class,Geography,Instrument Type,Asset Class Tag,ZSc,HalfLife
BPMR65Z Index,BNPP MarFA USD 30s50s Z,Rates,US,curve,rates_US_30s50s_curve,-0.8,1.8
BPML113Z Index,BNPP MarFA China 5s10s (IRS) Z,Rates,China,irs_curve,rates_China_5s10s_curve,-0.5,1.8
BPML124Z Index,BNPP MarFA China 2s5s10s (IRS) Z,Rates,China,irs_butterfly,rates_China_2s5s10s_butterfly,0.4,3.4
BPML105Z Index,BNPP MarFA India 5s10s (IRS) Z,Rates,India,irs_curve,rates_India_5s10s_curve,0.3,4.1
BPMR89Z Index,BNPP MarFA USD 5y5y Inflation Z,Rates,US,inflation_5y5y,rates_US_inflation_5y5y,-0.4,4.5
...,...,...,...,...,...,...,...
BPML78Z Index,BNPP MarFA Chile 2s10s (IRS) Z,Rates,Chile,irs_curve,rates_Chile_2s10s_curve,1.8,21.5
BPML139Z Index,BNPP MarFA China 2y (IRS) Z,Rates,China,irs_level,rates_China_2y_level,-1.9,22.2
BPML131Z Index,BNPP MarFA Hungary 2y (IRS) Z,Rates,Hungary,irs_level,rates_Hungary_2y_level,-1.8,22.2
BPML130Z Index,BNPP MarFA Czech 2y (IRS) Z,Rates,Czech,irs_level,rates_Czech_2y_level,-2.8,23.4


In [16]:
df.iloc[:40]

,Description,Asset Class,Geography,Instrument Type,Asset Class Tag,ZSc,HalfLife
BPMR65Z Index,BNPP MarFA USD 30s50s Z,Rates,US,curve,rates_US_30s50s_curve,-0.8,1.8
BPML113Z Index,BNPP MarFA China 5s10s (IRS) Z,Rates,China,irs_curve,rates_China_5s10s_curve,-0.5,1.8
BPML124Z Index,BNPP MarFA China 2s5s10s (IRS) Z,Rates,China,irs_butterfly,rates_China_2s5s10s_butterfly,0.4,3.4
BPML105Z Index,BNPP MarFA India 5s10s (IRS) Z,Rates,India,irs_curve,rates_India_5s10s_curve,0.3,4.1
BPMR89Z Index,BNPP MarFA USD 5y5y Inflation Z,Rates,US,inflation_5y5y,rates_US_inflation_5y5y,-0.4,4.5
BPMR72Z Index,BNPP MarFA USD 5s10s30s Z,Rates,US,butterfly,rates_US_5s10s30s_butterfly,-0.5,4.5
BPML120Z Index,BNPP MarFA Brazil 2s5s10s (IRS) Z,Rates,Brazil,irs_butterfly,rates_Brazil_2s5s10s_butterfly,0.9,4.6
BPML50Z Index,BNPP MarFA Korea 10y (IRS) spread Z,Rates,Korea,irs_level_spread,rates_Korea_10y_level_spread,-0.4,4.6
BPMR98Z Index,BNPP MarFA USD 5y5y real yield Z,Rates,US,real_yield_5y5y,rates_US_real_yield_5y5y,0.2,4.8
BPMR71Z Index,BNPP MarFA JPY 2s5s10s Z,Rates,Japan,butterfly,rates_Japan_2s5s10s_butterfly,-1.8,5.0


In [19]:
df[df['Asset Class']=='Commodity']

,Description,Asset Class,Geography,Instrument Type,Asset Class Tag,ZSc,HalfLife
BPMCO9Z Index,"BNPP MarFA Oil, Brent Z",Commodity,Global,energy,commodity_Oil_Brent,-1.8,10.9
BPMCO10Z Index,"BNPP MarFA Oil, WTI Z",Commodity,Global,energy,commodity_Oil_WTI,-1.8,11.3
BPMCO11Z Index,BNPP MarFA Gasoil Z,Commodity,Global,energy,commodity_Gasoil,-1.3,13.0
BPMCO6Z Index,BNPP MarFA Zinc Z,Commodity,Global,base_metal,commodity_Zinc,-1.6,13.3
BPMCO7Z Index,BNPP MarFA Aluminium Z,Commodity,Global,base_metal,commodity_Aluminium,-3.4,13.7
BPMCO12Z Index,BNPP MarFA Gasoline Z,Commodity,Global,energy,commodity_Gasoline,0.5,13.9
BPMCO5Z Index,BNPP MarFA Copper Z,Commodity,Global,base_metal,commodity_Copper,-0.3,14.5
BPMCO2Z Index,BNPP MarFA Platinum Z,Commodity,Global,precious_metal,commodity_Platinum,-0.6,14.9
BPMCO13Z Index,BNPP MarFA Natural Gas Z,Commodity,Global,energy,commodity_Natural_Gas,-1.3,16.1
BPMCO4Z Index,BNPP MarFA Palladium Z,Commodity,Global,precious_metal,commodity_Palladium,-1.1,16.5


In [20]:
df.iloc[-30:]

,Description,Asset Class,Geography,Instrument Type,Asset Class Tag,ZSc,HalfLife
BPME29Z Index,BNPP MarFA CSI 500 Z,Equity,China,index,equity_China_index,0.9,17.3
BPME30Z Index,BNPP MarFA CSI 1000 Z,Equity,China,index,equity_China_index,0.4,17.5
BPML73Z Index,BNPP MarFA Hungary 2s10s (IRS) Z,Rates,Hungary,irs_curve,rates_Hungary_2s10s_curve,0.5,17.6
BPML111Z Index,BNPP MarFA Thailand 5s10s (IRS) Z,Rates,Thailand,irs_curve,rates_Thailand_5s10s_curve,0.2,17.6
BPML135Z Index,BNPP MarFA Brazil 2y (IRS) Z,Rates,Brazil,irs_level,rates_Brazil_2y_level,0.4,17.6
BPMR80Z Index,BNPP MarFA 10y BTP vs Bono Spread Z,Rates,Italy-Spain,cross_market_spread,rates_xmkt_BTP_Bono_10y,-1.9,17.8
BPML134Z Index,BNPP MarFA South Africa 2y (IRS) Z,Rates,South Africa,irs_level,rates_South Africa_2y_level,-0.6,17.9
BPMR5Z Index,BNPP MarFA EUR 2y Z,Rates,Eurozone,level,rates_Eurozone_2y_level,-1.9,18.1
BPML3Z Index,BNPP MarFA Hungary 5y (IRS) Z,Rates,Hungary,irs_level,rates_Hungary_5y_level,-1.5,18.2
BPMCO3Z Index,BNPP MarFA Silver Z,Commodity,Global,precious_metal,commodity_Silver,-1.0,18.4
